Data Quality: Invalid Transactions

Banking scenario

Your transaction_fact Bronze data can contain bad records.

For this exercise, consider a transaction invalid if:

- transaction_id is NULL
- customer_id is NULL
- amount is NULL
- amount <= 0
- transaction_ts is NULL
- status is not "COMPLETED"

The business wants:
- Valid records → continue to Silver
- Invalid records → quarantine
- We should retain the reason why the record was rejected


In [0]:
%python
from pyspark.sql import functions as F


BRONZE_PATH = "/mnt/usvikformula1dl/bronze"
transaction_bronze_path = f"{BRONZE_PATH}/transaction_fact"
transaction_df=spark.read.format("delta").load(f"{transaction_bronze_path}")
transaction_df.printSchema()

valid_transaction_df=transaction_df.filter(F.col("customer_id").isNotNull()).filter(F.col("amount").isNotNull()).filter(F.col("transaction_ts").isNotNull()).filter(F.col("amount") > 0).filter(F.col("status")=="COMPLETED")

quarantine_transaction_df=transaction_df.withColumn("rejection_reason",
                                                    F.concat_ws(
                                                        ",",
                                                        F.when(F.col("customer_id").isNull(),F.lit("Null_Customer_id")),
                                                        F.when(F.col("amount") <= 0,F.lit("Negative amount")),
                                                        F.when(F.col("transaction_ts").isNull(),F.lit("Null transaction_ts")),
                                                        F.when(F.col("transaction_id").isNull(),F.lit("Null transaction_id")),
                                                        F.when(F.col("status") !="COMPLETED",F.lit("Status not completed")),
                                                                )).filter(F.col("rejection_reason") != "")


display(quarantine_transaction_df)